<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/Re_implementation_of_LLM_Enhanced_Multi_Agent_Architecture_for_Conversation_Based_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Re-implementation: Multi-Agent Architecture for Conversation-Based Assessment (AIED 2025)

## 1. Key Insights & Findings from the Paper

This notebook operationalizes the architecture proposed by **Hou et al. (AIED 2025)** in *"An LLM-Enhanced Multi-Agent Architecture for Conversation-Based Assessment"*.

Traditional LLM-based tutoring systems often suffer from conflating assessment with dialogue generation, leading to generic feedback, conversational drift, or elevated student evaluation anxiety. The paper resolves this by applying **Evidence-Centered Design (ECD)** to separate the conversational persona from the underlying assessment engine.

### Core Conceptual Takeaways:

1. **Decoupled Responsibilities:** Student-facing dialogue entities must be decoupled from behind-the-scenes assessment state machines.
2. **Multi-Agent Division of Labor:** Combining an authoritative **Expert Agent** with a collaborative **Peer Agent** balances conceptual scaffolding with anxiety reduction.
3. **Deterministic Orchestration (The Watcher):** Rather than using an LLM to decide conversational routing (which introduces latency, cost, and non-determinism), a **non-LLM rule-based Watcher** controls turn allocation, safety directives, and prompt injection based on real-time formative signals.
4. **Structured Evidence Streams:** Turn-by-turn interactions are evaluated silently by a **Formative Assessor** to log discrete evidence items, which are later synthesized into a holistic report by a **Summative Assessor**.

---

## 2. System Architecture & Agent Workflow

```
                        +-----------------------------------+
                        |          Student Input            |
                        +-----------------------------------+
                                          |
                                          v
+-----------------------------------------------------------------------------------+
|                            BEHIND-THE-SCENES ASSESSOR                             |
|  Formative Assessor (LLM): Parses utterance -> JSON Evidence (Claim & Level)      |
+-----------------------------------------------------------------------------------+
                                          |
                                          v
+-----------------------------------------------------------------------------------+
|                              DETERMINISTIC WATCHER                                |
|  Watcher (Non-LLM Rule Engine): Evaluates evidence log & turn count               |
|  -> Decides Next Agent (Expert vs. Peer) & Injects Behavioral Directives          |
+-----------------------------------------------------------------------------------+
                                          |
                        +-----------------+-----------------+
                        |                                   |
                        v                                   v
        +-------------------------------+   +-------------------------------+
        |     STUDENT-FACING AGENT      |   |     STUDENT-FACING AGENT      |
        |    Expert Agent (LLM):        |   |      Peer Agent (LLM):        |
        | Probing & Scaffolded Questions|   | Encouragement & Co-Inquiry    |
        +-------------------------------+   +-------------------------------+
                        |                                   |
                        +-----------------+-----------------+
                                          |
                                          v
                        +-----------------------------------+
                        |          Student Output           |
                        +-----------------------------------+
                                          |
                               (On Session Completion)
                                          v
+-----------------------------------------------------------------------------------+
|                             FINAL SUMMATIVE DIAGNOSIS                             |
|  Summative Assessor (LLM): Synthesizes full evidence log into an ECD report       |
+-----------------------------------------------------------------------------------+

```

---

## 3. Implementation Approach & Technical Design

### A. Inference Engine & Model Routing

* **Groq API Acceleration:** Multi-agent dialogue pipelines require sub-second inference latencies to maintain realistic turn-taking. We use Groq's hardware acceleration running open models (`llama-3.3-70b-versatile`, `qwen-2.5-32b`, or `llama-3.1-8b-instant`).
* **Dynamic Model Fallback & Validation:** The setup automatically queries available models on the Groq endpoint, filters out audio/transcription models (e.g., Whisper), and selects the highest-tier active LLM endpoint.

### B. State Management & Data Schemas

* **Auditability via `@dataclass`:** `DialogueState` holds session history, turn counts, and the cumulative list of `EvidenceItem` objects.
* **Structured JSON Formatting:** The `FormativeAssessor` enforces JSON schema output (`json_object`) for consistent schema parsing (`observed_claim`, `competency_level`, `recommended_next_focus`).

### C. Watcher Logic (Rule-Based Routing)

The `Watcher` class functions deterministically without additional LLM call overhead:

* **Low Competency Trigger:** If the Formative Assessor flags `competency_level == "Low"`, the Watcher immediately routes the turn to the **Peer Agent** with a directive to reduce anxiety and offer simpler scaffolding.
* **Turn Alternation:** Under normal conditions, it alternates between the **Expert Agent** (for conceptual pushing) and the **Peer Agent** (for collaborative inquiry).

### D. Key Security & Secret Management

To prevent API key leakage in version control, the environment reads credentials securely from Google Colab's encrypted secret store (`google.colab.userdata`) rather than hardcoding plaintext tokens.

In [6]:
# Cell 1: Install Groq SDK and dependencies
!pip install -q groq pydantic

import os
import json
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field
from google.colab import userdata
from groq import Groq

# Pull secret securely from Colab environment
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Groq automatically picks up os.environ["GROQ_API_KEY"]
client = Groq()

# Fetch active chat completion models from Groq API
all_models = [m.id for m in client.models.list().data]

preferred_chat_models = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "qwen-2.5-32b",
    "llama3-70b-8192",
    "llama3-8b-8192"
]

MODEL_NAME = next((m for m in preferred_chat_models if m in all_models), None)
if not MODEL_NAME:
    chat_models = [m for m in all_models if "whisper" not in m.lower() and "vision" not in m.lower()]
    MODEL_NAME = chat_models[0] if chat_models else "llama-3.1-8b-instant"

print(f"Selected Active Groq Model: {MODEL_NAME}")

# --- 1. STATE & DATA MODELS ---
@dataclass
class EvidenceItem:
    turn: int
    speaker: str
    utterance: str
    observed_claim: str
    competency_level: str  # Low, Medium, High

@dataclass
class DialogueState:
    turn_count: int = 0
    current_topic: str = "Photosynthesis & Plant Respiration"
    history: List[Dict[str, str]] = field(default_factory=list)
    evidence_log: List[EvidenceItem] = field(default_factory=list)
    active_agent: str = "Expert"  # Expert or Peer

# --- 2. AGENT DEFINITIONS ---

class FormativeAssessor:
    """Behind-the-scenes real-time evidence extractor based on Evidence-Centered Design (ECD)."""
    SYSTEM_PROMPT = """You are a Formative Assessor in an Evidence-Centered Design (ECD) framework.
Analyze the student's latest turn in the conversation about science inquiry.
Extract fine-grained evidence regarding their conceptual understanding.
You MUST output a valid JSON object strictly matching this schema:
{
  "observed_claim": "brief description of demonstrated knowledge or misconception",
  "competency_level": "Low" or "Medium" or "High",
  "recommended_next_focus": "what concept needs probing next"
}"""

    def analyze_turn(self, turn_num: int, student_utterance: str) -> EvidenceItem:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": f"Student response: '{student_utterance}'"}
            ],
            temperature=0.2
        )
        res_data = json.loads(response.choices[0].message.content)
        return EvidenceItem(
            turn=turn_num,
            speaker="Student",
            utterance=student_utterance,
            observed_claim=res_data.get("observed_claim", ""),
            competency_level=res_data.get("competency_level", "Medium")
        )

class ExpertAgent:
    """Student-facing domain expert who guides learning and asks leading questions."""
    SYSTEM_PROMPT = """You are an Expert Science Tutor agent in a conversation-based assessment.
Your goal is to assess student understanding of science inquiry through leading, open-ended questions.
Keep responses concise (1-2 sentences), encouraging, and focused on the scientific process."""

    def respond(self, history: List[Dict[str, str]], directive: str) -> str:
        messages = [{"role": "system", "content": f"{self.SYSTEM_PROMPT}\nDirective from Watcher: {directive}"}]
        messages.extend(history[-4:])  # Context window
        resp = client.chat.completions.create(model=MODEL_NAME, messages=messages, temperature=0.7)
        return resp.choices[0].message.content

class PeerAgent:
    """Student-facing learning peer who collaborates and models inquiry."""
    SYSTEM_PROMPT = """You are a friendly Peer Learning Agent in a science conversation.
You are learning alongside the student. Ask curious questions or share a friendly thought to make the student comfortable.
Keep responses under 2 sentences."""

    def respond(self, history: List[Dict[str, str]], directive: str) -> str:
        messages = [{"role": "system", "content": f"{self.SYSTEM_PROMPT}\nDirective from Watcher: {directive}"}]
        messages.extend(history[-4:])
        resp = client.chat.completions.create(model=MODEL_NAME, messages=messages, temperature=0.8)
        return resp.choices[0].message.content

class Watcher:
    """Non-LLM Rule-Based Orchestrator controlling state, turn alternation, and directives."""
    def __init__(self):
        self.max_turns = 6

    def route_turn(self, state: DialogueState, last_evidence: Optional[EvidenceItem]) -> Dict[str, Any]:
        state.turn_count += 1

        # Rule: Alternate between Expert and Peer unless student shows low competency
        if last_evidence and last_evidence.competency_level == "Low":
            next_agent = "Peer"  # Peer steps in to lower anxiety
            directive = "The student seems unsure. Provide an encouraging thought or simpler question."
        elif state.turn_count % 2 == 0:
            next_agent = "Peer"
            directive = "Elaborate on the topic naturally as a fellow classmate."
        else:
            next_agent = "Expert"
            directive = "Ask a leading question to push the student's conceptual reasoning."

        return {"next_agent": next_agent, "directive": directive}

class SummativeAssessor:
    """Behind-the-scenes agent for overall session diagnosis."""
    SYSTEM_PROMPT = """You are a Summative Assessor in an ECD framework.
Review the complete session history and evidence log to produce a final diagnostic summary of student mastery."""

    def synthesize_session(self, state: DialogueState) -> str:
        evidence_str = "\n".join([f"Turn {e.turn}: [{e.competency_level}] {e.observed_claim}" for e in state.evidence_log])
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": f"Evidence Log:\n{evidence_str}"}
            ]
        )
        return resp.choices[0].message.content

# --- 3. SIMULATION RUNNER ---
def run_cba_simulation():
    state = DialogueState()
    watcher = Watcher()
    formative = FormativeAssessor()
    expert = ExpertAgent()
    peer = PeerAgent()
    summative = SummativeAssessor()

    print("=== AIED 2025 CBA MULTI-AGENT SIMULATION INITIALIZED ===")

    simulated_student_turns = [
        "Plants need sunlight and water to make food through photosynthesis.",
        "I think oxygen is produced, but I'm not sure what happens at night when there is no light.",
        "Maybe they store the energy or use respiration?"
    ]

    for student_input in simulated_student_turns:
        print(f"\n--- Turn {state.turn_count + 1} ---")
        print(f"Student: {student_input}")
        state.history.append({"role": "user", "content": student_input})

        # 1. Behind-the-Scenes Formative Analysis
        evidence = formative.analyze_turn(state.turn_count + 1, student_input)
        state.evidence_log.append(evidence)
        print(f"[Formative Assessor] Claim: '{evidence.observed_claim}' | Level: {evidence.competency_level}")

        # 2. Watcher Routing
        decision = watcher.route_turn(state, evidence)
        print(f"[Watcher] Routed to: {decision['next_agent']} Agent")

        # 3. Student-Facing Agent Response
        if decision['next_agent'] == "Expert":
            bot_reply = expert.respond(state.history, decision['directive'])
            speaker = "Expert Agent"
        else:
            bot_reply = peer.respond(state.history, decision['directive'])
            speaker = "Peer Agent"

        print(f"{speaker}: {bot_reply}")
        state.history.append({"role": "assistant", "content": bot_reply})

    # 4. Final Diagnostic Synthesis
    print("\n=== FINAL SUMMATIVE DIAGNOSIS ===")
    final_report = summative.synthesize_session(state)
    print(final_report)

# Run simulation
run_cba_simulation()

Selected Active Groq Model: qwen/qwen3.8-27b
=== AIED 2025 CBA MULTI-AGENT SIMULATION INITIALIZED ===

--- Turn 1 ---
Student: Plants need sunlight and water to make food through photosynthesis.
[Formative Assessor] Claim: 'Student correctly identifies the primary inputs (sunlight, water) and the process (photosynthesis) for plant food production, but omits carbon dioxide as a necessary reactant.' | Level: Medium
[Watcher] Routed to: Expert Agent
Expert Agent: That's a great start! If you were to design an experiment to prove that light is necessary for this process, what specific variable would you change between your test group and your control group?

--- Turn 2 ---
Student: I think oxygen is produced, but I'm not sure what happens at night when there is no light.
[Formative Assessor] Claim: 'Student correctly identifies oxygen as a product of the process but demonstrates a gap in understanding the dependency of the process on light, specifically regarding the cessation of oxygen pr